# 05a — Normalisation, **Full** branch (468 face landmarks → 1,759 features/frame)

**Goal.** Turn raw landmarks into signer-normalised sequences. This branch keeps the **whole 468-point face mesh**. It is one of two branches; the other (05b) differs only in the face block.

**Inputs → Outputs.** Notebook 03 arrays → `normalized_landmarks/full/…/*.npy` `(T, 1759)`, `normalized_metadata_full.csv`, `normalized_feature_schema_full.json`.

**Approach** (identical code in both branches, applied per clip)
- **Global frame:** origin = shoulder midpoint (pose 11/12), scale = shoulder width in x–y. Unusable frames take the sequence median; both tracks are smoothed with a 5-frame moving average.
- **Local hands:** an extra wrist-centred copy of each hand, scaled by the mean wrist-to-MCP distance (landmarks 5, 9, 13, 17), so hand *shape* is separate from hand *position*. It is zeroed wherever the hand was not detected.
- **Gap filling:** missing global coordinates are linearly interpolated (edges held). A modality never seen in a clip stays zero.
- **Presence masks:** 4 binary columns (face, pose, left, right) record the *original* detection, so a filled value is never mistaken for an observed one.

**Key design decisions**
- Raw image coordinates encode where a signer stands and how big they are → normalising to a body-centred frame makes the same sign from different people look alike, which is what signer independence needs.
- Everything is per clip and nothing is fitted across the dataset → no information leaks between training and test signers at this stage.
- The two branch notebooks differ in exactly three lines (`VARIANT`, `FACE_INDICES`, the width assert) → any downstream difference is attributable to the face block.

**Frame schema**

| block | full | reduced |
|---|---|---|
| face (global) | 468 lm → `[0:1404]` | 92 lm → `[0:276]` |
| pose | `[1404:1503]` | `[276:375]` |
| global hands (L, R) | `[1503:1629]` | `[375:501]` |
| local hands (L, R) | `[1629:1755]` | `[501:627]` |
| presence masks | `[1755:1759]` | `[627:631]` |
| **width** | **1,759** | **631** |

The 355-column non-face tail is identical in both branches. Notebook 06 verifies this byte for byte.

In [ ]:
# ============================================================
# CELL 1 - Configuration
# ============================================================
# >>> THE ONLY LINE THAT DIFFERS BETWEEN THE TWO BRANCHES <<<
VARIANT = "full"
# ============================================================

import hashlib
import json
import multiprocessing
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Bump this if the normalisation maths ever changes, so old and new outputs
# can never be silently mixed in the same experiment.
NORMALIZATION_VERSION = "phase5-v1"

OUTPUT_ROOT = Path("/kaggle/working/normalized_landmarks") / VARIANT
METADATA_PATH = Path(f"/kaggle/working/normalized_metadata_{VARIANT}.csv")
SCHEMA_PATH = Path(f"/kaggle/working/normalized_feature_schema_{VARIANT}.json")

# Shoulder detection jitters frame to frame. Smoothing the origin and scale
# stops that jitter being injected into every other landmark. 1 disables it.
SMOOTH_WINDOW = 5

# Fill frames where a body part was not detected. The mask keeps a record of
# what was real, so nothing is hidden from the model.
INTERPOLATE_GAPS = True

# Pure NumPy here, so fork-based workers are safe (unlike Phase 3).
NUM_WORKERS = min(4, os.cpu_count() or 1)
PROGRESS_EVERY = 250

# Fingerprint of everything EXCEPT the face choice. Both branches must print
# the same value; that is the evidence they were processed identically.
SHARED_SETTINGS = {
    "normalization_version": NORMALIZATION_VERSION,
    "origin": "mid_shoulder_pose_11_12",
    "scale": "shoulder_width_xy",
    "smooth_window": SMOOTH_WINDOW,
    "interpolate_gaps": INTERPOLATE_GAPS,
    "local_hand_copy": True,
    "presence_mask_columns": 4,
    "dtype": "float32",
}
PIPELINE_ID = hashlib.sha256(
    json.dumps(SHARED_SETTINGS, sort_keys=True).encode()
).hexdigest()[:12]

print(f"VARIANT: {VARIANT}")
print(f"Shared pipeline id: {PIPELINE_ID}   <- must match the other branch")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | workers {NUM_WORKERS}")

In [ ]:
# ============================================================
# CELL 2 - Find the Phase 3 output and check it is the complete one
# ============================================================

def find_shards():
    found = sorted(Path("/kaggle/input").glob("*/holistic_landmarks_1629/shard_*"))
    if not found:
        found = sorted(Path("/kaggle/input").glob("**/holistic_landmarks_1629/shard_*"))
    if not found:
        raise FileNotFoundError(
            "No Phase 3 output attached. Right panel -> + Add Input -> Notebooks "
            "-> Phase 3 - Landmark Extraction -> the 'final run' version."
        )
    return found


SHARD_DIRS = find_shards()
SOURCE_FILES = sorted(path for shard in SHARD_DIRS for path in shard.rglob("*.npy"))

print(f"Shard folders: {[str(p) for p in SHARD_DIRS]}")
print(f"Landmark files found: {len(SOURCE_FILES):,}")

if len(SOURCE_FILES) != 6000:
    print(
        f"\nWARNING: expected 6,000 files, found {len(SOURCE_FILES):,}.\n"
        "The usual cause is attaching 'first extraction' (5,306 files) instead "
        "of 'final run'. Both branches MUST use the same input, or the "
        "comparison is meaningless. Fix this before continuing."
    )

sample = np.load(SOURCE_FILES[0])
print(f"\nSample: {SOURCE_FILES[0].name} -> {sample.shape}, {sample.dtype}")
assert sample.shape[1] == 1629, f"Expected 1629 raw features, found {sample.shape[1]}"
print("Input feature width confirmed: 1629")

In [ ]:
# ============================================================
# CELL 3 - Feature layout and the normalisation maths
# ============================================================
# Raw Phase 3 layout, as landmark indices (543 landmarks x 3 coordinates):
#   face 0:468 | pose 468:501 | left hand 501:522 | right hand 522:543

FACE_LANDMARKS, POSE_LANDMARKS, HAND_LANDMARKS = 468, 33, 21
TOTAL_LANDMARKS = 543
FACE_SLICE = slice(0, 468)
POSE_SLICE = slice(468, 501)
LEFT_HAND_SLICE = slice(501, 522)
RIGHT_HAND_SLICE = slice(522, 543)

POSE_LEFT_SHOULDER, POSE_RIGHT_SHOULDER = 11, 12
HAND_WRIST = 0
HAND_MCP_JOINTS = [5, 9, 13, 17]

EPS = 1e-6
SMOOTH_WINDOW = 5

FACE_SUBSET = [
    0, 7, 13, 14, 17, 33, 37, 39, 40, 46, 52, 53, 55, 61, 63, 65, 66, 70, 78,
    80, 81, 82, 84, 87, 88, 91, 95, 105, 107, 133, 144, 145, 146, 153, 154,
    155, 157, 158, 159, 160, 161, 163, 173, 178, 181, 185, 191, 246, 249, 263,
    267, 269, 270, 276, 282, 283, 285, 291, 293, 295, 296, 300, 308, 310, 311,
    312, 314, 317, 318, 321, 324, 334, 336, 362, 373, 374, 375, 380, 381, 382,
    384, 385, 386, 387, 388, 390, 398, 402, 405, 409, 415, 466,
]


def to_landmarks(sequence):
    """(T, 1629) -> (T, 543, 3)."""
    return sequence.reshape(sequence.shape[0], TOTAL_LANDMARKS, 3)


def modality_presence(points):
    """A modality is absent in a frame when MediaPipe wrote an all-zero block."""
    return {
        "face": np.any(points[:, FACE_SLICE, :] != 0, axis=(1, 2)),
        "pose": np.any(points[:, POSE_SLICE, :] != 0, axis=(1, 2)),
        "left_hand": np.any(points[:, LEFT_HAND_SLICE, :] != 0, axis=(1, 2)),
        "right_hand": np.any(points[:, RIGHT_HAND_SLICE, :] != 0, axis=(1, 2)),
    }


def moving_average(values, window):
    """Edge-padded moving average along axis 0. window <= 1 is a no-op."""
    if window <= 1 or values.shape[0] < 2:
        return values
    window = min(window, values.shape[0])
    if window % 2 == 0:
        window -= 1
    if window <= 1:
        return values
    pad = window // 2
    padded = np.pad(values, [(pad, pad)] + [(0, 0)] * (values.ndim - 1), mode="edge")
    kernel = np.ones(window, dtype=np.float32) / window
    if values.ndim == 1:
        return np.convolve(padded, kernel, mode="valid").astype(np.float32)
    out = np.empty_like(values, dtype=np.float32)
    for column in range(values.shape[1]):
        out[:, column] = np.convolve(padded[:, column], kernel, mode="valid")
    return out


def reference_and_scale(points, pose_present, smooth_window=SMOOTH_WINDOW):
    """Mid-shoulder origin and shoulder-width scale, per frame.

    Shoulder width uses the xy plane only: MediaPipe's z is weakly calibrated
    and including it makes the scale jitter without adding information.
    Frames without a usable pose inherit the sequence median.
    """
    pose = points[:, POSE_SLICE, :]
    left = pose[:, POSE_LEFT_SHOULDER, :]
    right = pose[:, POSE_RIGHT_SHOULDER, :]

    reference = (left + right) / 2.0
    scale = np.linalg.norm(left[:, :2] - right[:, :2], axis=1)
    usable = pose_present & (scale > EPS)

    if not np.any(usable):
        frames = points.shape[0]
        return (
            np.zeros((frames, 3), dtype=np.float32),
            np.ones(frames, dtype=np.float32),
            usable,
        )

    reference[~usable] = np.median(reference[usable], axis=0)
    scale[~usable] = np.median(scale[usable])

    reference = moving_average(reference.astype(np.float32), smooth_window)
    scale = moving_average(scale.astype(np.float32), smooth_window)
    scale = np.maximum(scale, EPS)
    return reference, scale, usable


def apply_global(points, reference, scale, presence):
    """Translate to mid-shoulder, divide by shoulder width, re-zero absences."""
    out = (points - reference[:, None, :]) / scale[:, None, None]
    out = out.astype(np.float32)
    for name, block in (
        ("face", FACE_SLICE),
        ("pose", POSE_SLICE),
        ("left_hand", LEFT_HAND_SLICE),
        ("right_hand", RIGHT_HAND_SLICE),
    ):
        out[~presence[name], block, :] = 0.0
    return out


def interpolate_gaps(points, present):
    """Linearly interpolate absent frames; hold the nearest value at the edges."""
    frames = points.shape[0]
    known = np.flatnonzero(present)
    if known.size == 0 or known.size == frames:
        return points, 0

    grid = np.arange(frames)
    next_pos = np.clip(np.searchsorted(known, grid, side="left"), 0, known.size - 1)
    prev_pos = np.clip(np.searchsorted(known, grid, side="right") - 1, 0, known.size - 1)
    low, high = known[prev_pos], known[next_pos]

    span = (high - low).astype(np.float32)
    weight = np.where(span > 0, (grid - low) / np.maximum(span, 1.0), 0.0)
    weight = weight.astype(np.float32)[:, None, None]

    blended = points[low] * (1.0 - weight) + points[high] * weight
    filled = points.copy()
    filled[~present] = blended[~present]
    return filled, int((~present).sum())


def local_hand(hand, present):
    """Wrist-centred, hand-size-scaled copy: shape independent of body scale."""
    centred = hand - hand[:, HAND_WRIST:HAND_WRIST + 1, :]
    size = np.linalg.norm(centred[:, HAND_MCP_JOINTS, :], axis=2).mean(axis=1)

    usable = size > EPS
    if np.any(usable):
        size = np.where(usable, size, np.median(size[usable]))
    else:
        size = np.ones_like(size)

    out = (centred / np.maximum(size, EPS)[:, None, None]).astype(np.float32)
    out[~present] = 0.0
    return out


SIGNER_PATTERN = re.compile(r"^p(\d+)_([A-Za-z0-9]+)_")


def parse_signer(stem):
    """'p10_c_aam' -> ('p10', 'c'). Unparseable names return (None, None)."""
    match = SIGNER_PATTERN.match(stem)
    if not match:
        return None, None
    return f"p{int(match.group(1))}", match.group(2)


# The face selection. This is what makes this branch different.
FACE_INDICES = np.arange(FACE_LANDMARKS)
FACE_COUNT = len(FACE_INDICES)
FEATURE_WIDTH = (FACE_COUNT + POSE_LANDMARKS + 2 * HAND_LANDMARKS) * 3 + 126 + 4


def normalize_sequence(sequence, smooth_window=SMOOTH_WINDOW, interpolate=INTERPOLATE_GAPS):
    """(T, 1629) raw -> (T, FEATURE_WIDTH) normalised, plus per-video statistics."""
    points = to_landmarks(np.asarray(sequence, dtype=np.float32))
    frames = points.shape[0]

    presence = modality_presence(points)
    reference, scale, pose_usable = reference_and_scale(
        points, presence["pose"], smooth_window
    )
    normalized = apply_global(points, reference, scale, presence)

    filled = {name: 0 for name in presence}
    if interpolate:
        for name, block in (
            ("face", FACE_SLICE), ("pose", POSE_SLICE),
            ("left_hand", LEFT_HAND_SLICE), ("right_hand", RIGHT_HAND_SLICE),
        ):
            normalized[:, block, :], filled[name] = interpolate_gaps(
                normalized[:, block, :], presence[name]
            )

    left_local = local_hand(normalized[:, LEFT_HAND_SLICE, :], presence["left_hand"])
    right_local = local_hand(normalized[:, RIGHT_HAND_SLICE, :], presence["right_hand"])

    mask = np.stack(
        [presence["face"], presence["pose"],
         presence["left_hand"], presence["right_hand"]],
        axis=1,
    ).astype(np.float32)

    features = np.concatenate(
        [
            normalized[:, FACE_INDICES, :].reshape(frames, -1),
            normalized[:, POSE_SLICE, :].reshape(frames, -1),
            normalized[:, LEFT_HAND_SLICE, :].reshape(frames, -1),
            normalized[:, RIGHT_HAND_SLICE, :].reshape(frames, -1),
            left_local.reshape(frames, -1),
            right_local.reshape(frames, -1),
            mask,
        ],
        axis=1,
    ).astype(np.float32)

    statistics = {
        "frames": frames,
        "pose_usable_frames": int(pose_usable.sum()),
        "pose_never_detected": bool(pose_usable.sum() == 0),
        "median_shoulder_width_px": float(np.median(scale)),
    }
    for name in ("face", "pose", "left_hand", "right_hand"):
        statistics[f"{name}_detected_frames"] = int(presence[name].sum())
        statistics[f"{name}_detection_rate"] = float(presence[name].mean())
        statistics[f"{name}_interpolated_frames"] = int(filled[name])

    return features, statistics


print(f"Face landmarks kept: {FACE_COUNT}")
print(f"Features per frame:  {FEATURE_WIDTH}")
assert FEATURE_WIDTH == 1759, "Feature width does not match this branch"

In [ ]:
# ============================================================
# CELL 4 - Write the schema: what every column means
# ============================================================
blocks, cursor = [], 0
for name, count, note in (
    ("face", FACE_COUNT, "global: mid-shoulder origin, shoulder-width scale"),
    ("pose", POSE_LANDMARKS, "global"),
    ("left_hand", HAND_LANDMARKS, "global"),
    ("right_hand", HAND_LANDMARKS, "global"),
    ("left_hand_local", HAND_LANDMARKS, "wrist origin, hand-size scale"),
    ("right_hand_local", HAND_LANDMARKS, "wrist origin, hand-size scale"),
):
    blocks.append({"block": name, "landmarks": count, "start": cursor,
                   "end_exclusive": cursor + count * 3, "note": note})
    cursor += count * 3
blocks.append({
    "block": "presence_mask", "landmarks": 4, "start": cursor,
    "end_exclusive": cursor + 4,
    "note": "1 = detected by MediaPipe, 0 = interpolated. "
            "Order: face, pose, left_hand, right_hand",
})
cursor += 4
assert cursor == FEATURE_WIDTH

SCHEMA = {
    "variant": VARIANT,
    "pipeline_id": PIPELINE_ID,
    "shared_settings": SHARED_SETTINGS,
    "features_per_frame": FEATURE_WIDTH,
    "dtype": "float32",
    "coordinate_order": "xyz per landmark",
    "face_landmarks_kept": FACE_COUNT,
    "face_indices": [int(index) for index in FACE_INDICES],
    "fitted_statistics": "none - every operation is geometric and per-video, so "
                         "no information crosses the signer split",
    "blocks": blocks,
}
SCHEMA_PATH.write_text(json.dumps(SCHEMA, indent=2), encoding="utf-8")

print(f"--- {VARIANT}: {FEATURE_WIDTH} features ---")
for block in blocks:
    print(f"  {block['start']:>5}:{block['end_exclusive']:<5} {block['block']}")
print(f"\nSchema written to {SCHEMA_PATH}")

In [ ]:
# ============================================================
# CELL 5 - Look at one real video before doing all 6,000
# ============================================================
raw = np.load(SOURCE_FILES[0])
features, statistics = normalize_sequence(raw)

print(f"File:    {SOURCE_FILES[0].name}")
print(f"Label:   {SOURCE_FILES[0].parent.name}")
print(f"Signer:  {parse_signer(SOURCE_FILES[0].stem)}")
print(f"raw:     {raw.shape}  ->  normalised: {features.shape}")

print("\nPer-video statistics:")
for key, value in statistics.items():
    print(f"  {key}: {value}")

pose_start = FACE_COUNT * 3
pose_points = features[:, pose_start:pose_start + 99].reshape(-1, POSE_LANDMARKS, 3)
widths = np.linalg.norm(
    pose_points[:, POSE_LEFT_SHOULDER, :2] - pose_points[:, POSE_RIGHT_SHOULDER, :2], axis=1
)
midpoints = (pose_points[:, POSE_LEFT_SHOULDER, :] + pose_points[:, POSE_RIGHT_SHOULDER, :]) / 2

print(f"\nShoulder width after scaling:      {widths.mean():.6f}  (target 1.0)")
print(f"Mid-shoulder distance from origin: {np.abs(midpoints).max():.2e}  (target 0)")
print(f"Raw range:        [{raw.min():.3f}, {raw.max():.3f}]")
print(f"Normalised range: [{features.min():.3f}, {features.max():.3f}]")
print(f"All finite: {np.isfinite(features).all()}")

In [ ]:
# ============================================================
# CELL 6 - Self-test. This must pass before the full run.
# ============================================================
# Two invented signers of different size, standing in different places, doing
# the identical sign. If normalisation works, their features come out equal.

failures = []


def check(name, condition, detail=""):
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")
    if not condition:
        failures.append(name)


def synthetic(frames=30, offset=(0.5, 0.5, 0.0), scale=0.2, seed=0, drop_left=()):
    generator = np.random.default_rng(seed)
    canonical = generator.normal(0, 0.3, size=(frames, TOTAL_LANDMARKS, 3)).astype(np.float32)
    canonical[:, POSE_SLICE.start + POSE_LEFT_SHOULDER] = [-0.5, 0.0, 0.0]
    canonical[:, POSE_SLICE.start + POSE_RIGHT_SHOULDER] = [0.5, 0.0, 0.0]
    world = canonical * scale + np.array(offset, dtype=np.float32)
    for index in drop_left:
        world[index, LEFT_HAND_SLICE, :] = 0.0
    return world.reshape(frames, -1)


small = synthetic(offset=(0.30, 0.40, 0.00), scale=0.18, seed=7)
large = synthetic(offset=(0.75, 0.62, 0.10), scale=0.42, seed=7)
a, _ = normalize_sequence(small)
b, _ = normalize_sequence(large)

check("same sign, different signer -> identical features",
      np.abs(a - b).max() < 1e-3, f"max diff {np.abs(a - b).max():.2e}")
check("raw features genuinely differed",
      np.abs(small - large).max() > 0.1, f"max diff {np.abs(small - large).max():.3f}")

gapped, gap_stats = normalize_sequence(synthetic(drop_left=(5, 6, 7)))
mask = gapped[:, -4:]
left_start = (FACE_COUNT + POSE_LANDMARKS) * 3
check("mask marks undetected frames", np.all(mask[[5, 6, 7], 2] == 0.0))
check("mask marks detected frames", mask[0, 2] == 1.0)
check("gaps filled rather than left at origin",
      np.abs(gapped[[5, 6, 7], left_start:left_start + 63]).sum() > 0)
check("interpolated frames counted", gap_stats["left_hand_interpolated_frames"] == 3)

one_gap, _ = normalize_sequence(synthetic(frames=11, drop_left=(5,)))
check("interpolation is a genuine blend",
      np.allclose(one_gap[5, left_start:left_start + 63],
                  one_gap[[4, 6], left_start:left_start + 63].mean(axis=0), atol=1e-5))

real, _ = normalize_sequence(np.load(SOURCE_FILES[0]))
check("real file has the right width", real.shape[1] == FEATURE_WIDTH, str(real.shape))
check("real file has no NaN or Inf", np.isfinite(real).all())

no_pose = synthetic(frames=12)
no_pose = no_pose.reshape(12, TOTAL_LANDMARKS, 3)
no_pose[:, POSE_SLICE, :] = 0.0
survived, survived_stats = normalize_sequence(no_pose.reshape(12, -1))
check("survives a video where pose was never found",
      np.isfinite(survived).all() and survived_stats["pose_never_detected"])

print()
if failures:
    raise RuntimeError(f"Self-test failed: {failures}. Do not run the full pass.")
print("Self-test clean. Save Version -> Save & Run All.")

In [ ]:
# ============================================================
# CELL 7 - Normalise every video
# ============================================================

def process_one(source_path_string: str) -> dict:
    source_path = Path(source_path_string)
    shard_root = next(p for p in source_path.parents if p.name.startswith("shard_"))
    relative = source_path.relative_to(shard_root)

    signer, session = parse_signer(source_path.stem)
    row = {
        "source_file": str(source_path),
        "relative_path": str(relative),
        "label": source_path.parent.name,
        "clip": source_path.stem,
        "signer_id": signer,
        "session": session,
        "variant": VARIANT,
    }

    try:
        features, statistics = normalize_sequence(np.load(source_path))

        destination = OUTPUT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        # np.save appends ".npy" to any path not already ending in it, so write
        # through an open handle to keep the temporary name exact.
        temporary = destination.parent / (destination.name + ".tmp")
        with open(temporary, "wb") as handle:
            np.save(handle, features, allow_pickle=False)
        os.replace(temporary, destination)

        row["normalized_file"] = str(destination)
        row["features"] = int(features.shape[1])
        row.update(statistics)
        row["status"] = "ok"
    except Exception as error:
        row["status"] = "failed"
        row["error_type"] = type(error).__name__
        row["error_message"] = str(error)

    return row


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
paths = [str(path) for path in SOURCE_FILES]
started = time.time()
records = []

if NUM_WORKERS > 1:
    with multiprocessing.get_context("fork").Pool(NUM_WORKERS) as pool:
        for index, record in enumerate(pool.imap_unordered(process_one, paths, chunksize=8), 1):
            records.append(record)
            if index % PROGRESS_EVERY == 0 or index == len(paths):
                elapsed = time.time() - started
                rate = index / max(elapsed, 1e-6)
                print(f"[{index:,}/{len(paths):,}] {elapsed / 60:.1f} min | "
                      f"{rate:.1f} videos/s | eta {(len(paths) - index) / rate / 60:.1f} min",
                      flush=True)
else:
    for index, path in enumerate(paths, 1):
        records.append(process_one(path))
        if index % PROGRESS_EVERY == 0 or index == len(paths):
            print(f"[{index:,}/{len(paths):,}]", flush=True)

normalized_metadata = pd.DataFrame(records).sort_values("relative_path").reset_index(drop=True)
failed = normalized_metadata[normalized_metadata["status"] != "ok"]

print(f"\nProcessed {len(normalized_metadata):,} videos in "
      f"{(time.time() - started) / 60:.1f} minutes")
print(f"Failed: {len(failed):,}")
if not failed.empty:
    print(failed[["clip", "error_type", "error_message"]].head(10).to_string())

In [ ]:
# ============================================================
# CELL 8 - Metadata, signer inventory, quality report
# ============================================================
normalized_metadata.to_csv(METADATA_PATH, index=False)
good = normalized_metadata[normalized_metadata["status"] == "ok"]

if good.empty:
    raise RuntimeError(
        "Every video failed. The error column above says why; nothing "
        "below would be meaningful."
    )

print("=" * 70)
print(f"DATASET  ({VARIANT} branch)")
print("=" * 70)
print(f"Videos:  {len(good):,}")
print(f"Labels:  {good['label'].nunique()}")
print(f"Signers: {good['signer_id'].nunique()}")
print(f"Frames:  {int(good['frames'].sum()):,} total | {good['frames'].mean():.1f} mean "
      f"| {int(good['frames'].min())}-{int(good['frames'].max())} range")

unparsed = good[good["signer_id"].isna()]
if not unparsed.empty:
    print(f"\nWARNING: {len(unparsed):,} clips have no readable signer id. "
          "A signer-independent split needs every clip attributed.")
    print(unparsed["clip"].head().to_string())

print("\n" + "=" * 70)
print("SIGNERS  (the unit the Phase 6 split must respect)")
print("=" * 70)
print(
    good.groupby("signer_id")
    .agg(clips=("clip", "size"), labels=("label", "nunique"))
    .sort_values("clips", ascending=False)
    .to_string()
)

print("\n" + "=" * 70)
print("DETECTION QUALITY  (fraction of frames MediaPipe actually found each part)")
print("=" * 70)
for part in ("face", "pose", "left_hand", "right_hand"):
    rate = good[f"{part}_detection_rate"]
    print(f"  {part:<12} mean {rate.mean():.3f} | median {rate.median():.3f} | "
          f"under 50%: {(rate < 0.5).sum():,} | frames filled: "
          f"{int(good[f'{part}_interpolated_frames'].sum()):,}")

print("\n" + "=" * 70)
print("FLAGGED VIDEOS  (candidates for exclusion in Phase 6)")
print("=" * 70)
no_pose = good[good["pose_never_detected"] == True]
no_hands = good[(good["left_hand_detection_rate"] == 0) &
                (good["right_hand_detection_rate"] == 0)]
print(f"  Pose never detected: {len(no_pose):,}")
print(f"  Neither hand ever detected: {len(no_hands):,}")
if not no_pose.empty:
    print(no_pose[["clip", "label", "signer_id"]].head(10).to_string())

print(f"\nMetadata written to {METADATA_PATH}")

In [ ]:
# ============================================================
# CELL 9 - Verify what landed on disk, then summarise
# ============================================================
files = sorted(OUTPUT_ROOT.rglob("*.npy"))
problems = []

widths, finite = set(), True
for index in np.unique(np.linspace(0, len(files) - 1, min(30, len(files))).astype(int)):
    array = np.load(files[index])
    widths.add(array.shape[1])
    finite &= bool(np.isfinite(array).all())

total_bytes = sum(path.stat().st_size for path in files)

if len(files) != len(good):
    problems.append(f"{len(files)} files written, expected {len(good)}")
if widths != {FEATURE_WIDTH}:
    problems.append(f"widths {sorted(widths)}, expected {FEATURE_WIDTH}")
if not finite:
    problems.append("non-finite values found in the sample")

print(json.dumps({
    "variant": VARIANT,
    "pipeline_id": PIPELINE_ID,
    "files": len(files),
    "features_per_frame": FEATURE_WIDTH,
    "observed_widths": sorted(widths),
    "all_finite_in_sample": finite,
    "size_gib": round(total_bytes / (1024 ** 3), 2),
}, indent=2))

loaded = np.load(files[0])
mask = loaded[:, -4:]
print(f"\nRound trip: {files[0].name} -> {loaded.shape}")
print(f"  mask is strictly 0/1: {set(np.unique(mask).tolist()) <= {0.0, 1.0}}")
print(f"  detected fractions (face, pose, L, R): {mask.mean(axis=0).round(3).tolist()}")

working = sum(p.stat().st_size for p in Path("/kaggle/working").rglob("*") if p.is_file())
print(f"\n/kaggle/working total: {working / (1024 ** 3):.2f} GB of the 20 GB cap")

print("\n" + "=" * 70)
if problems:
    print("PROBLEMS FOUND:")
    for problem in problems:
        print(f"  - {problem}")
else:
    print(f"PHASE 5 COMPLETE - {VARIANT} branch")
    print("=" * 70)
    print(f"  normalized_landmarks/{VARIANT}/   {len(files):,} files, "
          f"{FEATURE_WIDTH} features per frame")
    print(f"  {METADATA_PATH.name}")
    print(f"  {SCHEMA_PATH.name}")
    print()
    print(f"  Pipeline id: {PIPELINE_ID}")
    print("  This must match the other branch. If it does not, the two were")
    print("  processed differently and the comparison is not valid.")
    print()
    print("  Phase 6: split by signer_id, never randomly. Fit any z-scoring on")
    print("  the training signers only.")
print("=" * 70)